# J2 Après-Midi | > Prompt engineering

**Objectifs d'apprentissage :**
1. Prise en main l'API d'un Modèle de Langage.
2. Maîtriser les principes du *Prompt Engineering* (Zero-shot, Few-shot, Chain of Thought).
3. Comparer les performances (VADER vs Modèle Local HuggingFace vs API LLM).
4. Mettre en place une validation humaine (inter-coder agreement) pour vérifier les résultats.
5. Appliquer ces concepts pour annoter le jeu de données INA (collecté au module précédent).


## 1. Introduction à l'API OpenAI

Lorsqu'on utilise ChatGPT via le site web, l'interface graphique envoie en arrière-plan nos requêtes à l'API d'OpenAI. Nous allons automatiser cela directement en Python.

**Attention !** En temps normal, ne mettez **jamais** votre clé API directement dans le code public. Pour les besoins de cet atelier, vous allez copier-coller temporairement la clé fournie par les instructeurs directement dans la variable `api_key`.

In [ ]:
from openai import OpenAI
import pandas as pd

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except:
    print("Erreur d'initialisation. Vérifiez votre clé API.")


In [ ]:
# Warm-up
text = "Peux-tu m'expliquer le prompt engineering très simplement ?"

client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                # {"role": "system", "content": "Vous êtes un assistant utile."},
                {"role": "user", "content": text}
            ],
            # temperature 0 = prévisible/déterministe, 1 = créatif/aléatoire
            temperature=1,  
            max_tokens=100
        )


### Hack-Time 🛠️

Essayez d'afficher uniquement le texte. 

# Votre code 

### Nous pouvons créer une fonction que nous allons réutiliser plus tard.

In [ ]:

def analyze_text(text, system_prompt, model="gpt-4o-mini", temperature=0.0):
    """
    Fonction simple pour envoyer un texte et un prompt système à l'API OpenAI.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            # temperature 0 = prévisible/déterministe, 1 = créatif/aléatoire
            temperature=temperature,  
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Erreur lors de l'appel à l'API : {e}"


## 2. L'impact du Prompt Engineering sur la Performance

En sciences sociales, **le prompt est l'équivalent d'un protocole d'annotation (codebook)**. Si le prompt est ambigu, le codage de vos données sera de mauvaise qualité.

Pour observer concrètement l'impact de l'amélioration des prompts, nous allons utiliser un petit jeu de données de test avec 5 phrases politiques, complexes, à analyser (ironie, négation, nuance).
Nous évaluerons si l'IA trouve le vrai label (Positif, Négatif ou Neutre).

In [ ]:
# Création de notre jeu de données
data_test = [
    {"text": "Le ministre n'a pas exactement brillé par son courage politique lors de la réforme.", "label": "NEGATIF"},
    {"text": "La nouvelle loi a été adoptée à l'Assemblée Nationale, le vote a eu lieu mardi à 15h.", "label": "NEUTRE"},
    {"text": "Bravo au gouvernement pour cette augmentation d'impôts insupportable, on adore !", "label": "NEGATIF"},
    {"text": "C'est un véritable triomphe historique pour l'opposition qui a su imposer ses idées brillantes.", "label": "POSITIF"},
    {"text": "Malgré quelques bonnes idées au départ, le projet final est un désastre économique.", "label": "NEGATIF"}
]

df_test = pd.DataFrame(data_test)
df_test


### A. Les approches classiques et locales (Rappels)

Testons d'abord **VADER** (basé sur un dictionnaire de mots) et le modèle local **DistilCamembert** (vu précédemment).

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()
analyseur_local = pipeline("sentiment-analysis", model="cmarkea/distilcamembert-base-sentiment")

def evaluer_vader(texte):
    score = sia.polarity_scores(texte)['compound']
    if score > 0.1: return "POSITIF"
    elif score < -0.1: return "NEGATIF"
    else: return "NEUTRE"

def evaluer_hf(texte):
    # Le modèle retourne par exemple [{'label': '1 star', 'score': 0.8}]
    # On va simplifier les étoiles (1-2 négatif, 3 neutre, 4-5 positif)
    res = analyseur_local(texte)[0]['label']
    if '1' in res or '2' in res: return "NEGATIF"
    elif '4' in res or '5' in res: return "POSITIF"
    else: return "NEUTRE"

df_test["vader"] = df_test["text"].apply(evaluer_vader)
df_test["camembert"] = df_test["text"].apply(evaluer_hf)

df_test[["text", "label", "vader", "camembert"]]


> 💡 **Observation** : VADER échoue souvent sur l'ironie car il compte juste les mots positifs (ex: "Bravo", "adore"). Le modèle local s'en sort un peu mieux mais peut faire des erreurs sur des nuances très spécifiques. Voyons ce qu'apporte le LLM avec différents niveaux de prompt.

### B. Zero-Shot Prompting
C'est la méthode la plus simple. On donne l'instruction sans fournir d'exemple préalable.

In [ ]:
zero_shot_prompt = """
Analyse le sentiment politique du texte suivant (POSITIF, NEGATIF, NEUTRE).
Réponds UNIQUEMENT par la catégorie, sans ponctuation.
"""

df_test["zero_shot"] = df_test["text"].apply(lambda x: analyze_text(x, zero_shot_prompt))


### C. Few-Shot Prompting
Fournir des exemples (*Few-Shot*) agit comme un **livre de codage (codebook)** pour l'IA.

In [ ]:
few_shot_prompt = """
Analyse le sentiment politique du texte (POSITIF, NEGATIF, NEUTRE).
Attention au sarcasme et à l'ironie.

Exemples :
- Texte : "Quel génie d'avoir détruit notre industrie !"
  Catégorie : NEGATIF
- Texte : "Une réunion aura lieu demain."
  Catégorie : NEUTRE

À toi :
Réponds UNIQUEMENT par la catégorie.
"""

df_test["few_shot"] = df_test["text"].apply(lambda x: analyze_text(x, few_shot_prompt))


### D. Chain of Thought (Chaîne de Pensée)
Demander au modèle de justifier son choix *avant* de donner la réponse finale l'aide à mieux gérer l'ironie et les retournements de situation.

In [ ]:
cot_prompt = """
Analyse le sentiment politique du texte (POSITIF, NEGATIF, NEUTRE).
Procède ainsi :
1. Raisonnement : Analyse le ton, cherche l'ironie ou la contradiction.
2. Label : Termine ta réponse par exactement : 'LABEL: [Catégorie]'.
"""

def extract_label(cot_response):
    if "LABEL: POSITIF" in cot_response.upper(): return "POSITIF"
    if "LABEL: NEGATIF" in cot_response.upper(): return "NEGATIF"
    if "LABEL: NEUTRE" in cot_response.upper(): return "NEUTRE"
    return "ERREUR"

df_test["cot_reasoning"] = df_test["text"].apply(lambda x: analyze_text(x, cot_prompt))
df_test["cot"] = df_test["cot_reasoning"].apply(extract_label)


### E. Comparaison des performances

Calculons le pourcentage de bonnes réponses (accuracy) pour chaque approche.

In [ ]:
# Calcul des performances
colonnes_modeles = ["vader", "camembert", "zero_shot", "few_shot", "cot"]
performances = {}

for col in colonnes_modeles:
    bonnes_reponses = sum(df_test[col] == df_test["label"])
    accuracy = (bonnes_reponses / len(df_test)) * 100
    performances[col] = f"{accuracy}%"

print("--- Accuracy par méthode ---")
for modele, acc in performances.items():
    print(f"{modele} : {acc}")

# Affichage des erreurs pour comprendre
df_test[["text", "label"] + colonnes_modeles]


### Hack-Time 🛠️

Observez le DataFrame ci-dessus. 
1. Sur quelle phrase le Zero-Shot s'est-il trompé par rapport au Few-Shot ou CoT ?
2. **Exercice :** Modifiez le `few_shot_prompt` ou le `cot_prompt` dans les cellules précédentes pour voir si vous pouvez obtenir 100% de réussite. Testez avec d'autres phrases !

## 3. Validation Humaine (Inter-coder Reliability)

Attention à ne jamais aveuglément faire confiance à un modèle... Il est indispensable de coder humainement un sous-échantillon (par ex. 10%) pour vérifier que vous êtes d'accord avec l'IA.

Nous allons vous demander de coder 3 phrases manuellement. Ensuite, nous les ferons coder par l'IA et nous calculerons votre accord.

In [ ]:
# 1. Vos données à coder
phrases_a_coder = [
    "La réforme des retraites est un désastre social sans précédent.",
    "Le texte a été voté par 349 voix pour et 86 contre.",
    "Une avancée majeure pour les droits des travailleurs a été obtenue hier soir."
]

# 2. Entrez vos labels humains ici ! 
# Remplacez "..." par "POSITIF", "NEGATIF" ou "NEUTRE"
mes_labels_humains = [
    "NEGATIF", # Pour la phrase 1
    "NEUTRE",  # Pour la phrase 2
    "POSITIF"  # Pour la phrase 3
]

# 3. L'IA code les mêmes phrases
labels_ia = []
for phrase in phrases_a_coder:
    # On utilise le prompt Zero-Shot pour l'exemple
    rep = analyze_text(phrase, zero_shot_prompt)
    labels_ia.append(rep)

# 4. Comparaison
df_validation = pd.DataFrame({
    "text": phrases_a_coder,
    "humain": mes_labels_humains,
    "ia": labels_ia
})
df_validation["Accord ?"] = df_validation["humain"] == df_validation["ia"]
df_validation

In [ ]:

taux_accord = df_validation['Accord ?'].mean() * 100
print(f"Taux d'accord Humain vs IA : {taux_accord}%")


> 💡 **Méthodologie** : Dans un vrai papier de recherche, on utiliserait le Kappa de Cohen ou le Krippendorff's Alpha pour mesurer cet accord de façon plus robuste statistiquement.

## 4. Atelier de Codage : Le Projet INA / ZFE

Appliquons maintenant ce que nous avons appris sur notre cas d'usage réel : les "Zones à Faibles Émissions" (ZFE) sur CNews et TF1.

In [41]:
df_ina = pd.read_csv("https://raw.githubusercontent.com/mickaeltemporao/lillelms/refs/heads/main/data/ina_zfe/ina_zfe.csv")
df_ina.head()


,indice,chaine,date,heure,duree,titre,lien,emission,type
0,1,TF1,14/03/2023,10:00:00,00:04:25,ZFE : les nouvelles règles,http://ina.fr/1,Le 13H,Débat
1,2,TF1,06/03/2023,12:45:00,00:05:55,Point d'étape sur les ZFE,http://ina.fr/2,Sept à Huit,Reportage
2,3,TF1,25/03/2023,13:00:00,00:01:27,Les vignettes Crit'Air en débat,http://ina.fr/3,Sept à Huit,Débat
3,4,TF1,17/10/2023,17:00:00,00:04:24,Les vignettes Crit'Air en débat,http://ina.fr/4,Le 20H,Plateau
4,5,TF1,24/05/2023,09:30:00,00:07:12,Écologie punitive : faut-il supprimer les ZFE ?,http://ina.fr/5,Sept à Huit,Débat


Notre objectif de recherche : **Le titre de l'émission est-il favorable, défavorable, ou neutre vis-à-vis de la politique environnementale des ZFE ?**

In [42]:
system_prompt_zfe = """
Tu es un assistant de recherche en sociologie des médias.
Ton but est d'annoter l'orientation d'un titre de journal télévisé vis-à-vis des politiques environnementales (ZFE).

Classe le titre selon 3 catégories :
- FAVORABLE : Le titre présente les ZFE comme une bonne chose, une solution, un progrès.
- DEFAVORABLE : Le titre souligne la colère, les problèmes, les interdictions, les coûts ou le côté punitif.
- NEUTRE : Le titre est purement descriptif et factuel, sans jugement de valeur.

Réponds UNIQUEMENT par la catégorie (FAVORABLE, DEFAVORABLE, NEUTRE).
"""


### Hack-Time 🛠️

Le code ci-dessous parcourt les 5 premières lignes de notre DataFrame INA.
1. Complétez la boucle pour envoyer chaque titre de l'émission à l'API OpenAI avec notre `system_prompt_zfe`.
2. Observez les résultats. Êtes-vous d'accord avec l'IA ?

In [43]:
resultats_ina = []

for index, row in df_ina.head(5).iterrows():
    titre = row['titre']
    chaine = row['chaine']
    
    # Appel à l'API
    # N'oubliez pas d'utiliser la fonction analyze_text
    annotation = analyze_text(titre, system_prompt_zfe)
    
    print(f"[{chaine}] {titre} \n-> {annotation}\n")
    resultats_ina.append(annotation)


[TF1] ZFE : les nouvelles règles 
-> NEUTRE

[TF1] Point d'étape sur les ZFE 
-> NEUTRE

[TF1] Les vignettes Crit'Air en débat 
-> NEUTRE

[TF1] Les vignettes Crit'Air en débat 
-> NEUTRE

[TF1] Écologie punitive : faut-il supprimer les ZFE ? 
-> DEFAVORABLE



## Conclusion

Nous avons vu comment utiliser le Prompt Engineering pour améliorer la qualité de l'annotation automatique, et comment valider ces résultats.

**Principes de recherche à retenir :**
1. **Tester différentes approches** : Zero-Shot, Few-Shot, CoT. Choisissez celles qui maximisent les performances sur un échantillon.
2. **Biais du Modèle :** Les modèles comportent les mêmes biais qu'on voit autour de nous. Attention à ne pas reproduire cela à grande échelle.
3. **Reproductibilité :** Si l'API met à jour son modèle, vos résultats dans 6 mois pourraient différer. Conservez toujours vos prompts exacts et les versions des modèles (ex: `gpt-4o-mini-2024-07-18`) dans vos scripts !
4. **Validation Intercondeur:** Ne par faire confiance... L'étape de codage humain sur un échantillon est le minimum pour la rigueur scientifique.